# Agregando caminos ramificados

Dado que la mayoría de los sistemas del mundo real no son lineales, es necesario tener rutas con ramificaciones. Por tanto, pasaremeos de este modelo

![](images/example_simplest_model.png)

O este modelo

![](images/example_simple_model_sequential.png)

A un modelo de este tipo:

![](images/example_simple_model_branching.png)

Para modelar una **ruta ramificada** dentro de una simulación DES, podemos recurrir a un recurso clásico de Python: la **lógica condicional**.

En muchos casos, las bifurcaciones en un modelo DES se basan en **probabilidades** que representan la proporción de entidades (por ejemplo, pacientes) que siguen una determinada trayectoria dentro del sistema. Por ejemplo, los datos podrían indicar que el **60% de los pacientes** consultan a un médico después de haber sido atendidos por una enfermera.

Para representar este comportamiento, podemos **muestrear aleatoriamente un número de una distribución uniforme** entre 0 y 1 y **compararlo con la probabilidad establecida**. Si el valor obtenido es menor que dicha probabilidad, asumimos que el paciente sigue esa ruta.

Esta técnica es efectiva porque traduce de manera simple la probabilidad teórica en una decisión binaria dentro del modelo, manteniendo la coherencia estadística del sistema simulado.


El **60% de los valores comprendidos entre 0 y 1** son menores que 0.6.

Por lo tanto, si todos los valores tienen la **misma probabilidad de ser seleccionados** —como ocurre en una distribución uniforme—, existe una **probabilidad del 60%** de obtener un valor inferior a 0.6.

Podemos aprovechar este principio para **simular la probabilidad de que una entidad siga una ruta específica dentro del modelo**.

---

No todas las rutas ramificadas se basan en probabilidades.

En algunos casos, las trayectorias pueden depender de otros factores, tales como:

* La **hora del día**.
* El **tipo de paciente** o entidad.
* El **tiempo** que una entidad permanece en una actividad determinada.
* Entre otros.

En estos escenarios, seguimos utilizando **lógica condicional**, pero modificamos la condición que se evalúa.

Por ejemplo:

* Si la decisión depende de la hora del día, deberíamos consultar el **tiempo actual de la simulación**.
* Si depende del tipo de paciente, dicha información podría almacenarse en un **atributo** del objeto paciente.

---

### Clase `glo`

Necesitamos añadir algunos parámetros adicionales a nuestra clase `glo`.


In [ ]:
# Clase para almacenar los valores globales de los parámetros.  
# No se crea una instancia de esta clase; simplemente se hace referencia
# al plano (blueprint) de la clase para acceder a los valores definidos en su interior.
class g:
    patient_inter = 5
    mean_reception_time = 2
    mean_n_consult_time = 6
    mean_d_consult_time = 20  ##Aquí
    number_of_receptionists = 1
    number_of_nurses = 1
    number_of_doctors = 2  ##Aquí
    prob_seeing_doctor = 0.6  ##Aquí
    sim_duration = 120
    number_of_runs = 5



### Clase `Patient`

Queremos añadir un atributo adicional para registrar el tiempo que los pacientes pasan con el médico si llegan a verlo.


In [ ]:
# Clase que representa a los pacientes que llegan a la clínica.
class Patient:
    def __init__(self, p_id):
        self.id = p_id
        self.q_time_recep = 0
        self.q_time_nurse = 0
        self.q_time_doctor = 0  ##Aquí

### Clase del modelo

#### Método `__init__`

En el método de inicialización, añadimos algunos atributos adicionales para almacenar nuevas salidas del modelo.


In [ ]:
# Clase que representa nuestro modelo de la clínica. 
class Model:
    # Constructor que configura el modelo para una ejecución.
    # Se pasa un número de ejecución (run number) al crear un nuevo modelo.
    def __init__(self, run_number):
        # Crear un entorno de SimPy en el cual se desarrollará toda la simulación
        self.env = simpy.Environment()

        # Crear un contador de pacientes (que usaremos como identificador del paciente)
        self.patient_counter = 0

        # Crear los recursos del modelo
        self.receptionist = simpy.Resource(
            self.env, capacity=g.number_of_receptionists
        )
        self.nurse = simpy.Resource(self.env, capacity=g.number_of_nurses)
        self.doctor = simpy.Resource(
            self.env, capacity=g.number_of_doctors)  ##Aquí

        # Almacenar el número de ejecución recibido
        self.run_number = run_number

        # Crear un nuevo DataFrame de Pandas que almacenará algunos resultados
        # asociados al ID del paciente (que usaremos como índice)
        self.results_df = pd.DataFrame()
        self.results_df["Patient ID"] = [1]
        self.results_df["Q Time Recep"] = [0.0]
        self.results_df["Time with Recep"] = [0.0]
        self.results_df["Q Time Nurse"] = [0.0]
        self.results_df["Time with Nurse"] = [0.0]
        self.results_df["Q Time Doctor"] = [0.0]  ##Aquí
        self.results_df["Time with Doctor"] = [0.0]  ##Aquí
        self.results_df.set_index("Patient ID", inplace=True)

        # Crear un atributo para almacenar los tiempos promedio de espera en cola
        # durante esta ejecución del modelo
        self.mean_q_time_recep = 0
        self.mean_q_time_nurse = 0
        self.mean_q_time_doctor = 0  ##Aquí

#### Método `generator_patient_arrivals`

Este método no cambia.

#### Método `attend_clinic`

Aquí debemos añadir una probabilidad de que los pacientes vean al médico durante su recorrido.


In [ ]:
def attend_clinic(self, patient):
    start_q_recep = self.env.now

    with self.receptionist.request() as req:
        yield req

        end_q_recep = self.env.now

        patient.q_time_recep = end_q_recep - start_q_recep

        sampled_recep_act_time = random.expovariate(
            1.0 / g.mean_reception_time
        )

        self.results_df.at[patient.id, "Q Time Recep"] = (
             patient.q_time_recep
        )
        self.results_df.at[patient.id, "Time with Recep"] = (
             sampled_recep_act_time
        )

        yield self.env.timeout(sampled_recep_act_time)

    # Aquí es donde el paciente termina con la recepcionista y comienza
    # a hacer fila para la enfermera

    start_q_nurse = self.env.now

    with self.nurse.request() as req:
        yield req

        end_q_nurse = self.env.now

        patient.q_time_nurse = end_q_nurse - start_q_nurse

        sampled_nurse_act_time = random.expovariate(1.0 /
                                                    glo.mean_n_consult_time)

        self.results_df.at[patient.id, "Q Time Nurse"] = (
            patient.q_time_nurse)
        self.results_df.at[patient.id, "Time with Nurse"] = (
            sampled_nurse_act_time)

        yield self.env.timeout(sampled_nurse_act_time)

        # Cuando el tiempo anterior finaliza, la función generadora regresará aquí.
        # Como no hay más instrucciones escritas, la función simplemente terminará.
        # Este es un punto de salida (sink).

    ## -----------------------------------------------------------
    ## Aquí comienza nuestro nuevo código para la consulta con el médico.
    ## Usamos lógica condicional para determinar si el paciente pasa
    ## a ver al médico o no.
    ## ------------------------------------------------------------
    #
    # Muestreamos un valor de una distribución uniforme entre 0 y 1.
    # Si el valor es menor que la probabilidad de ver a un médico (almacenada en la clase g),
    # entonces consideramos que el paciente consulta con el médico.
    #
    # Si no, este bloque de código no se ejecutará y el paciente
    # simplemente saldrá del sistema (podríamos agregar un “else” si
    # quisiéramos modelar una rama alternativa hacia otra actividad).

    if random.uniform(0, 1) < g.prob_seeing_doctor:
        start_q_doctor = self.env.now

        with self.doctor.request() as req:
            yield req

            end_q_doctor = self.env.now

            patient.q_time_doctor = end_q_doctor - start_q_doctor

            sampled_doctor_act_time = random.expovariate(
                1.0 / g.mean_d_consult_time
            )

            self.results_df.at[patient.id, "Q Time Doctor"] = (
                patient.q_time_doctor
            )
            self.results_df.at[patient.id, "Time with Doctor"] = (
                sampled_doctor_act_time
            )

            yield self.env.timeout(sampled_doctor_act_time)



Intentemos entender un poco mejor cómo activamos la lógica condicional.

Veamos la salida de la línea `random.uniform(0,1)`.


In [ ]:
import random

random.seed(42)
random.uniform(0,1)

¿Qué pasa si lo ejecutamos varias veces?



In [ ]:
for i in range(10):
  print(random.uniform(0,1))


¿Cómo se relaciona esto con nuestro código?

En la clase `glo`, establecemos un umbral de probabilidad para que los pacientes sean atendidos por el médico:


In [ ]:
print(glo.prob_seeing_doctor)

El código de la clase **Model** comprueba si el número generado por el generador de números aleatorios es menor que el umbral de probabilidad. Si lo es, el paciente pasa a ver al **doctor**; de lo contrario, ha llegado al final de su recorrido y abandona el sistema (**un sumidero**).



In [ ]:
for i in range(10):
  random_number = random.uniform(0,1)
  is_below_threshold = random_number < g.prob_seeing_doctor

  if is_below_threshold:
    print(f"Random number {random_number:.2f} is LOWER than threshold ({g.prob_seeing_doctor}). " +
    "Doctor code is triggered.")
  else:
    print(f"Random number {random_number:.2f} is HIGHER than threshold ({g.prob_seeing_doctor}). " +
    "Doctor code is **not** triggered.")


Si ejecutamos este código cien mil veces y graficamos el resultado, vemos que aparece un patrón a pesar del elemento aleatorio del generador de números.


In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

random_vals = [random.uniform(0,1) for i in range(100000)]

random_vals_df = pd.DataFrame({"value" :random_vals})

random_vals_df['threshold'] = np.where(random_vals_df["value"]<0.6, 'below', 'above')

fig = px.histogram(random_vals_df, color="threshold")

fig.update_traces(xbins=dict(
        start=0.0,
        end=1.0,
        size=0.1
    ),
    marker_line_width=1,marker_line_color="black")


fig.show()



Así, por cada 1000 pacientes, *aproximadamente* 600 verán a un médico y *aproximadamente* 400 abandonarán el sistema justo después de ver a la enfermera.

#### Método `calculate_run_results`

En este método, añadimos un paso adicional para medir el **tiempo medio de cola del médico** entre todos los pacientes de esta ejecución.


In [ ]:
# Este método calcula los resultados de una sola ejecución.  
# Aquí solo calculamos una media, pero en modelos reales probablemente se querría calcular más indicadores.
def calculate_run_results(self):
    # Calcular la media de los tiempos de espera en cola de los pacientes en esta ejecución del modelo.
    self.mean_q_time_recep = self.results_df["Q Time Recep"].mean()
    self.mean_q_time_nurse = self.results_df["Q Time Nurse"].mean()
    self.mean_q_time_doctor = self.results_df["Q Time Doctor"].mean()  ##Aquí


#### Método `run`

El método `run` no cambia.

### Clase `Trial`

#### Método `__init__`

En el método de inicialización, añadimos un simple marcador para medir el tiempo medio de cola del médico.


In [ ]:
def  __init__(self):
    self.df_trial_results = pd.DataFrame()
    self.df_trial_results["Run Number"] = [0]
    self.df_trial_results["Mean Q Time Recep"] = [0.0]
    self.df_trial_results["Mean Q Time Nurse"] = [0.0]
    self.df_trial_results["Mean Q Time Doctor"] = [0.0] ##Aqui
    self.df_trial_results.set_index("Run Number", inplace=True)


#### Método `run_trial`

Aquí simplemente añadimos el **tiempo medio de cola del médico** al *dataframe* de resultados del ensayo.


In [ ]:
def run_trial(self):
    # Ejecutar la simulación el número de veces especificado en la clase g.
    # Para cada ejecución, se crea una nueva instancia de la clase Model y se llama a su
    # método run, lo que pone todo el sistema en funcionamiento.
    # Una vez que la ejecución ha finalizado, se extraen los resultados almacenados
    # (en este caso, los tiempos promedio de espera en cola) y se guardan junto con
    # el número de ejecución en el DataFrame de resultados del experimento.
    for run in range(g.number_of_runs):
        my_model = Model(run)
        my_model.run()

        ##Aquí - se agregó el tiempo promedio de espera en cola para el médico al final de la lista
        self.df_trial_results.loc[run] = [my_model.mean_q_time_recep,
                                          my_model.mean_q_time_nurse,
                                          my_model.mean_q_time_doctor] #Aqui

    # Una vez que el experimento (es decir, todas las ejecuciones) ha finalizado,
    # se imprimen los resultados finales.
    self.print_trial_results()



## El código completo

In [ ]:
import simpy 
import random
import pandas as pd

# Clase para almacenar los valores de los parámetros globales.
# No se crea una instancia de esta clase; simplemente se hace referencia
# al plano (blueprint) de la clase para acceder a los valores definidos en su interior.
class g:
    patient_inter = 5
    mean_reception_time = 2
    mean_n_consult_time = 6
    mean_d_consult_time = 20  ##Aquí
    number_of_receptionists = 1
    number_of_nurses = 1
    number_of_doctors = 2  ##Aquí
    prob_seeing_doctor = 0.6  ##Aquí
    sim_duration = 120
    number_of_runs = 1

# Clase que representa a los pacientes que llegan a la clínica.
class Patient:
    def __init__(self, p_id):
        self.id = p_id
        self.q_time_recep = 0
        self.q_time_nurse = 0
        self.q_time_doctor = 0  ##Aquí

# Clase que representa nuestro modelo de la clínica.
class Model:
    # Constructor que configura el modelo para una ejecución.
    # Se pasa un número de ejecución (run number) al crear un nuevo modelo.
    def __init__(self, run_number):
        # Crear un entorno de SimPy en el cual se desarrollará toda la simulación
        self.env = simpy.Environment()

        # Crear un contador de pacientes (que usaremos como identificador del paciente)
        self.patient_counter = 0

        # Crear los recursos del modelo
        self.receptionist = simpy.Resource(
            self.env, capacity=g.number_of_receptionists
        )
        self.nurse = simpy.Resource(self.env, capacity=g.number_of_nurses)
        self.doctor = simpy.Resource(
            self.env, capacity=g.number_of_doctors)  ##Aquí

        # Almacenar el número de ejecución recibido
        self.run_number = run_number

        # Crear un nuevo DataFrame de Pandas que almacenará algunos resultados
        # asociados al ID del paciente (que usaremos como índice)
        self.results_df = pd.DataFrame()
        self.results_df["Patient ID"] = [1]
        self.results_df["Q Time Recep"] = [0.0]
        self.results_df["Time with Recep"] = [0.0]
        self.results_df["Q Time Nurse"] = [0.0]
        self.results_df["Time with Nurse"] = [0.0]
        self.results_df["Q Time Doctor"] = [0.0]  ##Aquí
        self.results_df["Time with Doctor"] = [0.0]  ##Aquí
        self.results_df.set_index("Patient ID", inplace=True)

        # Crear un atributo para almacenar los tiempos promedio de espera en cola
        # durante esta ejecución del modelo
        self.mean_q_time_recep = 0
        self.mean_q_time_nurse = 0
        self.mean_q_time_doctor = 0  ##Aquí

    # Función generadora que representa el generador DES de llegadas de pacientes
    def generator_patient_arrivals(self):
        # Usamos un bucle infinito para continuar generando pacientes mientras
        # la simulación esté en ejecución
        while True:
            # Incrementar el contador de pacientes en 1 (el primer paciente tendrá ID = 1)
            self.patient_counter += 1

            # Crear un nuevo paciente (instancia de la clase Patient definida arriba)
            # Se pasa el contador como ID del paciente.
            p = Patient(self.patient_counter)

            # Indicar a SimPy que inicie la función generadora attend_clinic con este paciente.
            # Esta función modelará el recorrido del paciente a través del sistema.
            self.env.process(self.attend_clinic(p))

            # Muestrear aleatoriamente el tiempo hasta la llegada del siguiente paciente.
            # Se usa una distribución exponencial (común para tiempos entre llegadas)
            # y se pasa un valor lambda = 1 / media. La media está almacenada en la clase g.
            sampled_inter = random.expovariate(1.0 / g.patient_inter)

            # Pausar esta función hasta que haya transcurrido el tiempo entre llegadas muestreado.
            # Nota: el tiempo en SimPy progresa en “Unidades de Tiempo”, que pueden representar
            # cualquier unidad (solo asegúrate de ser consistente en todo el modelo).
            yield self.env.timeout(sampled_inter)

    # Función generadora que representa el recorrido de un paciente a través de la clínica.
    # El objeto paciente se pasa a la función generadora para poder extraer y registrar información.
    def attend_clinic(self, patient):
        start_q_recep = self.env.now

        with self.receptionist.request() as req:
            yield req

            end_q_recep = self.env.now

            patient.q_time_recep = end_q_recep - start_q_recep

            sampled_recep_act_time = random.expovariate(
                1.0 / g.mean_reception_time
            )

            self.results_df.at[patient.id, "Q Time Recep"] = (
                 patient.q_time_recep
            )
            self.results_df.at[patient.id, "Time with Recep"] = (
                 sampled_recep_act_time
            )

            yield self.env.timeout(sampled_recep_act_time)

        # Aquí es donde el paciente termina con la recepcionista y comienza
        # a hacer fila para la enfermera

        # Registrar el momento en que el paciente empezó a hacer cola para la enfermera
        start_q_nurse = self.env.now

        # Este bloque solicita un recurso de enfermera y ejecuta el código dentro
        # de la sección con ese recurso asignado (no disponible para otros pacientes)
        with self.nurse.request() as req:
            # Pausar la función hasta que la enfermera esté disponible.
            # El paciente se encuentra actualmente en cola.
            yield req

            # Cuando llegamos aquí, la solicitud de enfermera ha sido satisfecha.
            # Ahora tenemos a la enfermera y hemos dejado de esperar,
            # por lo que registramos la hora actual como fin de la espera.
            end_q_nurse = self.env.now

            # Calcular el tiempo que este paciente esperó por la enfermera
            # y almacenarlo en su atributo correspondiente.
            patient.q_time_nurse = end_q_nurse - start_q_nurse

            # Muestrear aleatoriamente el tiempo que el paciente pasa con la enfermera.
            # Se usa una distribución exponencial por simplicidad, aunque en modelos reales
            # se suele emplear una distribución log-normal.
            sampled_nurse_act_time = random.expovariate(1.0 /
                                                        g.mean_n_consult_time)

            # Guardar el tiempo de espera y el tiempo de atención con la enfermera
            # en el DataFrame de resultados.
            self.results_df.at[patient.id, "Q Time Nurse"] = (
                patient.q_time_nurse)
            self.results_df.at[patient.id, "Time with Nurse"] = (
                sampled_nurse_act_time)

            # Pausar la función durante el tiempo de consulta muestreado.
            yield self.env.timeout(sampled_nurse_act_time)

            # Cuando este tiempo termina, la función generadora regresa aquí.
            # Como no hay más instrucciones, la función termina. Este es un punto de salida (sink).

        ##Aquí - Se agregó lógica condicional para determinar si el paciente pasa a ver al médico.
        # Se toma una muestra de una distribución uniforme entre 0 y 1.
        # Si el valor es menor que la probabilidad de ver a un médico (almacenada en la clase g),
        # entonces el paciente consultará con el médico.
        # Si no, este bloque no se ejecutará y el paciente simplemente saldrá del sistema.
        if random.uniform(0,1) < g.prob_seeing_doctor:
            start_q_doctor = self.env.now

            with self.doctor.request() as req:
                yield req

                end_q_doctor = self.env.now

                patient.q_time_doctor = end_q_doctor - start_q_doctor

                sampled_doctor_act_time = random.expovariate(
                    1.0 / g.mean_d_consult_time
                )

                self.results_df.at[patient.id, "Q Time Doctor"] = (
                    patient.q_time_doctor
                )
                self.results_df.at[patient.id, "Time with Doctor"] = (
                    sampled_doctor_act_time
                )

                yield self.env.timeout(sampled_doctor_act_time)

    # Este método calcula los resultados de una sola ejecución.
    # Aquí solo calculamos una media, pero en modelos reales probablemente se calcularían más indicadores.
    def calculate_run_results(self):
        # Calcular la media de los tiempos de espera en cola de los pacientes en esta ejecución del modelo.
        self.mean_q_time_recep = self.results_df["Q Time Recep"].mean()
        self.mean_q_time_nurse = self.results_df["Q Time Nurse"].mean()
        self.mean_q_time_doctor = self.results_df["Q Time Doctor"].mean()  ##Aquí

    # El método run inicia los generadores de entidades DES, ejecuta la simulación
    # y luego llama a los métodos necesarios para calcular los resultados de la ejecución.
    def run(self):
        # Iniciar el generador DES que crea nuevos pacientes.
        # En este modelo solo hay uno, pero podrían iniciarse varios si existieran.
        self.env.process(self.generator_patient_arrivals())

        # Ejecutar el modelo durante la duración especificada en la clase g.
        self.env.run(until=g.sim_duration)

        # Una vez finalizada la simulación, calcular los resultados de la ejecución.
        self.calculate_run_results()

        # Imprimir el número de ejecución y los resultados a nivel de paciente.
        print(f"Run Number {self.run_number}")
        print(self.results_df)

# Clase que representa un conjunto de ejecuciones (Trial) de la simulación.
class Trial:
    # El constructor crea un DataFrame de pandas para almacenar los resultados clave
    # de cada ejecución, con el número de ejecución como índice.
    def  __init__(self):
        self.df_trial_results = pd.DataFrame()
        self.df_trial_results["Run Number"] = [0]
        self.df_trial_results["Mean Q Time Recep"] = [0.0]
        self.df_trial_results["Mean Q Time Nurse"] = [0.0]
        self.df_trial_results["Mean Q Time Doctor"] = [0.0]  ##Aquí
        self.df_trial_results.set_index("Run Number", inplace=True)

    # Método para imprimir los resultados del conjunto de ejecuciones.
    # En modelos reales, probablemente se guardarían además de imprimirlos.
    def print_trial_results(self):
        print("Trial Results")
        print(self.df_trial_results)

    # Método para ejecutar un conjunto de simulaciones (Trial)
    def run_trial(self):
        # Ejecutar la simulación el número de veces especificado en la clase g.
        # Para cada ejecución, se crea una nueva instancia de Model y se llama a su método run.
        # Una vez completada la ejecución, se almacenan los resultados (tiempos promedio de espera)
        # junto con el número de ejecución en el DataFrame de resultados.
        for run in range(g.number_of_runs):
            my_model = Model(run)
            my_model.run()

            ##Aquí - se agregó el tiempo promedio de espera en cola para el médico al final de la lista.
            self.df_trial_results.loc[run] = [my_model.mean_q_time_recep,
                                              my_model.mean_q_time_nurse,
                                              my_model.mean_q_time_doctor]

        # Una vez completado el conjunto de ejecuciones, imprimir los resultados finales.
        self.print_trial_results()


## Evaluación de los resultados

Analicemos los resultados obtenidos para una sola ejecución del modelo.

Cuando un paciente **no consulta con el médico**, observa que el valor correspondiente en esa fila aparece como `NaN`, que significa *“no es un número”*.
Este valor se trata de manera diferente a un `0` en los cálculos de la media: un `NaN` **no se incluye** en la operación, mientras que un tiempo en cola igual a `0` **sí influye** en el resultado promedio.

In [ ]:
# Crear una instancia de la clase Trial
my_trial = Trial()

# Llamar al método run_trial del objeto Trial
my_trial.run_trial()
